# Step 30 — cross-platform, use B: discover endotypes together, across platforms

**Data types: RNA_array + bulk_RNA_seq + scRNA_seq pseudobulk.** **Reads:** `step10_site_*`,
`step13_site_*`, `step13_model.rds`, `step26_shared.rds`, `step23_*`, `step25_*`.
**Writes:** `step30_cross_platform.rds`.

Five sites run the federated recipe of steps 11–13 together:

| site | study | data type | patients |
|---|---|---|---|
| A, B, C | GSE65391 | RNA_array | discovery patients, first visit |
| GSE232381 | GSE232381 | bulk_RNA_seq | all 16 |
| GSE135779 | GSE135779 | scRNA_seq pseudobulk | the 33 children with SLE |

**Genes:** the model's genes that every study measures (step 26). **Units:** each site standardises
every gene on its own patients (mean 0, SD 1). That is what makes a microarray site and a sequencing
site comparable, and it also removes any real difference in average expression between studies.
Studies differ in tissue and in which patients they recruited, so this is a strong assumption, and it
is stated here rather than hidden.

**The comparison:** the same recipe on the three array sites alone, in the same units. Adding the
RNA sequencing sites either leaves the array endotypes in place (ARI near 1 on the array patients) or
moves them.

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
source("../src/federation.R")
start_log("30")
sh <- readRDS(art("step26_shared.rds")); model <- readRDS(art("step13_model.rds"))
G <- intersect(model$genes, sh$shared)
length(G)
zsite <- function(E) t(scale(t(E[G, , drop = FALSE])))      # genes x patients, standardised at the site
core_of <- function(m) { d <- m[m$split == "discovery", ]; d <- d[order(d$subject, d$visit), ]; rownames(d)[!duplicated(d$subject)] }
Z <- list()
for (s in SITES) { d <- readRDS(site_file("10", s)); Z[[s]] <- zsite(d$E[, core_of(d$meta)]) }
b <- readRDS(art("step23_GSE232381.rds")); Z$GSE232381 <- zsite(b$E)
p <- readRDS(art("step25_GSE135779.rds")); Z$GSE135779 <- zsite(p$E[, p$meta$disease == "SLE"])
sapply(Z, ncol)

[1] 677

A         B         C GSE232381 GSE135779 
       37        37        36        16        33

In [2]:
discover <- function(sites, seed) {
  set.seed(seed)
  ps <- pool_suff(lapply(names(sites), function(s) send(site_suff(sites[[s]]), s, "per-gene sums", ncol(sites[[s]]))))
  pc <- federated_pca(sites, ps$mean, ps$n, k = 10, tol = 1e-10)
  P  <- lapply(sites, function(X) t(X - ps$mean) %*% pc$rotation)
  runs <- lapply(setNames(2:6, 2:6), function(k) federated_consensus(P, k, rep(0, 10), pc$sdev, n_resamples = 100))
  pac  <- sapply(runs, function(r) pac_from_hist(Reduce(`+`, r$histogram)))
  k    <- choose_k(data.frame(k = 2:6, pac = pac))
  fit  <- federated_kmeans_best(P, k, rep(0, 10), pc$sdev, nstart = 50)
  # gene-space centroids from per-cluster sums
  sums <- lapply(names(sites), function(s) { lab <- fit$labels[[s]]
    send(list(count = tabulate(lab, k), sum = t(sapply(seq_len(k), function(j) rowSums(sites[[s]][, lab == j, drop = FALSE])))),
         s, "per-cluster count and gene sums", ncol(sites[[s]])) })
  cen <- Reduce(`+`, lapply(sums, `[[`, "sum")) / Reduce(`+`, lapply(sums, `[[`, "count"))
  list(k = k, pac = pac, labels = fit$labels, size = fit$size, centroids = cen)
}
with_rna    <- discover(Z, SEED + 300L)
array_only  <- discover(Z[SITES], SEED + 300L)
rbind(with_rnaseq = with_rna$pac, array_only = array_only$pac)
c(k_with_rnaseq = with_rna$k, k_array_only = array_only$k)

,2,3,4,5,6
with_rnaseq,0.1984674,0.2908046,0.4551724,0.4371648,0.3885057
array_only,0.2079511,0.3603466,0.3929664,0.4087666,0.3802243


k_with_rnaseq  k_array_only 
            2             2

## Did the RNA sequencing sites move the array endotypes?

In [3]:
ari_array <- sapply(SITES, function(s) ari(with_rna$labels[[s]], array_only$labels[[s]]))
round(ari_array, 3)
# both runs against the step 13 endotypes of the same patients
orig <- lapply(SITES, function(s) { d <- readRDS(site_file("13", s)); as.character(d$meta[colnames(Z[[s]]), "endotype"]) })
round(c(with_rnaseq = ari(unlist(with_rna$labels[SITES]), unlist(orig)),
        array_only  = ari(unlist(array_only$labels[SITES]), unlist(orig))), 3)

A     B     C 
0.790 0.892 0.889

with_rnaseq  array_only 
      0.792       0.792

## Where do the RNA sequencing patients fall?

Each site reports its counts per endotype. Endotypes are matched to E1 and E2 by the sign of their
centroid on the E2 marker genes.

In [4]:
e2m <- intersect(sh$markers[31:60], G)
e2_side <- rowMeans(with_rna$centroids[, match(e2m, G), drop = FALSE])
name <- ifelse(e2_side > 0, "E2-like", "E1-like")
round(e2_side, 2)
counts <- t(sapply(names(Z), function(s) table(factor(name[with_rna$labels[[s]]], levels = c("E1-like", "E2-like")))))
counts
tab_b <- table(ln_activity = b$meta$ln_activity, endotype = factor(name[with_rna$labels$GSE232381], levels = c("E1-like", "E2-like")))
tab_b
fisher.test(tab_b)$p.value

[1]  0.54 -0.34

,E1-like,E2-like
A,22,15
B,22,15
C,20,16
GSE232381,12,4
GSE135779,21,12


           endotype
ln_activity E1-like E2-like
   inactive       5       1
   active         7       3

[1] 1

In [5]:
saveRDS(list(genes = G, with_rnaseq = with_rna, array_only = array_only, names = name,
             counts = counts, ari_array = ari_array), art("step30_cross_platform.rds"))

## Findings

- **Adding the RNA sequencing sites does not change the answer.** Both runs choose k = 2 (PAC 0.198
  with the sequencing sites, 0.208 without). On the array patients, the two runs agree closely (ARI
  0.79 to 0.89 by site), and both agree with the step 13 endotypes to the same degree (ARI 0.79). The
  remaining difference from step 13 comes from the per-site standardisation and the reduced gene set
  (677 genes), not from the new sites.
- **Both endotypes occur on every platform.** GSE232381 has 12 E1-like and 4 E2-like patients;
  GSE135779 has 21 and 12. In PBMC the E2-like group is defined mostly by lower lymphocyte genes, since
  the granulocyte genes are barely measured there (step 27).
- **Nephritis activity is again unrelated to endotype** in GSE232381 (Fisher p = 1.0).
- The sequencing sites add 49 patients to the 110 array patients without sharing a single value, on
  a different platform. With so few of them, they confirm the array endotypes rather than refine them.